# פרק 7: בניית אפליקציות צ'אט
## התחלה מהירה עם Microsoft Foundry Models API

פנקס זה מותאם מ-[מאגר דוגמאות Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) שכולל פנקסים שניגשים לשירותי [Azure OpenAI](notebook-azure-openai.ipynb).

> **הערה:** GitHub Models ייפסק בסוף יולי 2026. פנקס זה מכוון כעת ל-[Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), שמציע את אותו קטלוג מודלים חינמי לנסות וחוויית SDK להסקת מסקנות ב-Azure AI.


# סקירה  
"מודלים גדולים של שפה הם פונקציות שממירות טקסט לטקסט. בהינתן מחרוזת טקסט קלט, מודל שפה גדול מנסה לחזות את הטקסט שיבוא אחריו"(1). מחברת ה"Quickstart" הזו תציג למשתמשים מושגים ברמה גבוהה של מודלים גדולים של שפה, דרישות חבילה מרכזיות להתחלת עבודה עם AML, הצגה רכה לעיצוב פרומפט, וכמה דוגמאות קצרות של שימושים שונים. 


## תוכן עניינים  

[סקירה כללית](#overview)  
[כיצד להשתמש בשירות OpenAI](#how-to-use-openai-service)  
[1. יצירת שירות OpenAI שלך](#1.-creating-your-openai-service)  
[2. התקנה](#2.-installation)    
[3. אישורים](#3.-credentials)  

[מקרי שימוש](#use-cases)    
[1. סיכום טקסט](#1.-summarize-text)  
[2. סיווג טקסט](#2.-classify-text)  
[3. יצירת שמות מוצר חדשים](#3.-generate-new-product-names)  
[4. כוונון עדין של מסווג](#4.fine-tune-a-classifier)  

[הפניות](#references)


### בנה את ההנחיה הראשונה שלך  
תרגיל קצר זה יספק מבוא בסיסי להגשת הנחיות למודל ב-Microsoft Foundry Models למשימה פשוטה של "סיכום".


**שלבים**:  
1. התקן את הספרייה `azure-ai-inference` בסביבת הפייתון שלך, אם עדיין לא עשית זאת.  
2. טען ספריות עזר סטנדרטיות והגדר את האישורים שלך ב-Microsoft Foundry Models.  
3. בחר מודל למשימה שלך  
4. צור הנחיה פשוטה למודל  
5. הגש את הבקשה שלך ל-API של המודל!


### 1. התקן את `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. ייבא ספריות עזר ואתחל האישורים


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. מציאת המודל הנכון  
דגמים כמו GPT-4o ו-GPT-4o מיני יכולים להבין וליצור שפה טבעית, וזמינים בקטלוג דגמי Microsoft Foundry לצד דגמים מ-Meta, Mistral, Cohere, ומicrosoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. עיצוב פרומפט  

"הקסם של מודלים לשוניים גדולים הוא שבאמצעות אימון למזער את טעות החיזוי הזו על פני כמויות עצומות של טקסט, המודלים לומדים בסופו של דבר מושגים שימושיים לחיזויים אלה. לדוגמה, הם לומדים מושגים כמו"(1):

* איך לאיית
* איך עובדת הדקדוק
* איך לפרפרז
* איך לענות על שאלות
* איך לנהל שיחה
* איך לכתוב בשפות רבות
* איך לקודד
* וכו'.

#### איך לשלוט במודל לשוני גדול  
"מבין כל הקלטים למודל לשוני גדול, הכי משפיע הוא הפרומפט של הטקסט(1).

ניתן להניע מודלים לשוניים גדולים להפיק פלט בכמה דרכים:

הנחיה: אמור למודל מה אתה רוצה
השלמה: גרום למודל להשלים את תחילת מה שאתה רוצה
הדגמה: הראה למודל מה אתה רוצה, עם:
כמה דוגמאות בפרומפט
מאות או אלפי דוגמאות בערכת אימון לכיול עדין"



#### יש שלושה קווים מנחים בסיסיים ליצירת פרומפטים:

**הראה וספר**. הבהר מה אתה רוצה באמצעות הוראות, דוגמאות או שילוב של שניהם. אם אתה רוצה שהמודל ידרג רשימת פריטים בסדר אלפביתי או יסווג פסקה על פי סנטימנט, הראה לו שזה מה שאתה רוצה.

**ספק נתונים איכותיים**. אם אתה מנסה לבנות מסווג או לגרום למודל לעקוב אחרי תבנית, וודא שיש מספיק דוגמאות. ודא לעבור על הדוגמאות שלך — המודל חכם בדרך כלל מספיק לזהות טעויות איות בסיסיות ולספק תגובה, אבל הוא גם עשוי להניח שזה מכוון וזה יכול להשפיע על התגובה.

**בדוק את ההגדרות שלך.** הגדרות ה-temperature וה-top_p שולטות עד כמה המודל דטרמיניסטי ביצירת התגובה. אם אתה מבקש תגובה שבה יש רק תשובה נכונה אחת, אז תרצה להוריד את ההגדרות האלה. אם אתה מחפש תגובות מגוונות יותר, אז ייתכן שתרצה להעלות אותן. הטעות מספר אחד שאנשים עושים בהגדרות האלה היא להניח שהן שולטות ב"חוכמה" או ב"יצירתיות".


מקור: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. שלח!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### חזור על אותה הקריאה, איך התוצאות משוות? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## סיכום טקסט  
#### אתגר  
סכם טקסט על ידי הוספת 'tl;dr:' בסוף קטע הטקסט. שים לב כיצד המודל מבין כיצד לבצע מספר משימות ללא הוראות נוספות. באפשרותך להתנסות בהנחיות תיאוריות יותר מ- tl;dr כדי לשנות את התנהגות המודל ולהתאים אישית את הסיכום שתקבל(3).  

עבודה אחרונה הדגימה שיפורים משמעותיים בהרבה משימות וסטנדרטים של עיבוד שפה טבעית על ידי אימון מוקדם על אוסף טקסטים גדול ואחריו כיוונון מיטבי למשימה ספציפית. אף על פי שבדרך כלל הארכיטקטורה היא בלתי תלויה במשימה, שיטה זו עדיין דורשת מערכי נתוני כיוונון למשימות ספציפיות הכוללים אלפי או עשרות אלפי דוגמאות. לעומת זאת, בני אדם בדרך כלל יכולים לבצע משימת שפה חדשה רק עם כמה דוגמאות או הוראות פשוטות - דבר שמערכות עיבוד שפה טבעית עכשוויות עדיין נאבקות בו במידה רבה. כאן אנחנו מראים כי הגדלת מודלי השפה משפרת מאוד את הביצועים הבלתי תלויים במשימה עם מעט דוגמאות, לפעמים אף מגיעה לתחרותיות עם שיטות הכיוונון המיטבי המובילות עד כה.  



tl;dr  


# תרגילים למספר מקרים של שימוש  
1. לסכם טקסט  
2. לסווג טקסט  
3. לייצר שמות חדשים למוצרים


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## סיווג טקסט  
#### אתגר  
סווג פריטים לקטגוריות הנמסרות בזמן ההסקה. בדוגמה הבאה, אנו מספקים גם את הקטגוריות וגם את הטקסט לסיווג בהנחיה (*playground_reference). 

פניה מלקוח: שלום, אחת המפתחות במקלדת המחשב הנייד שלי נשברה לאחרונה ואצטרך החלפה:

קטגוריה מסווגת:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## יצירת שמות מוצרים חדשים
#### אתגר
צור שמות מוצרים מתוך מילים לדוגמה. כאן אנו כוללים בהוראה מידע על המוצר עבורו אנחנו הולכים ליצור שמות. אנו גם מספקים דוגמה דומה כדי להראות את הדפוס שאנו רוצים לקבל. כמו כן, הגדרנו ערך טמפרטורה גבוה כדי להגדיל את המקריות ואת התגובות החדשניות יותר.

תיאור מוצר: מכשיר ביתי להכנת מילקשייק
מילים בסיס: מהיר, בריא, קומפקטי.
שמות מוצרים: HomeShaker, Fit Shaker, QuickShake, Shake Maker

תיאור מוצר: זוג נעליים שיכולות להתאים לכל גודל רגל.
מילים בסיס: מתכוונן, מתאים, אומי-פיט.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# הפניות  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# לעזרה נוספת  
[צוות המסחור של OpenAI](AzureOpenAITeam@microsoft.com) 


# תורמים
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
